In [ ]:
!pip install -q langgraph langchain langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 7.7 MB/s eta 0:00:00


In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END


In [ ]:
class AgentState(TypedDict):
    user_query: str
    clarification: Optional[str]
    is_ambiguous: bool


In [ ]:
AMBIGUOUS_KEYWORDS = ["report", "design", "analysis", "plan"]

def detect_ambiguity(state: AgentState):
    query = state["user_query"].lower()
    for word in AMBIGUOUS_KEYWORDS:
        if word in query:
            return {
                "is_ambiguous": True,
                "clarification": None
            }
    return {
        "is_ambiguous": False,
        "clarification": None
    }


In [ ]:
def ask_clarification(state: AgentState):
    query = state["user_query"].lower()

    if "report" in query:
        question = "What type of report do you want? (sales / academic / project)"
    elif "design" in query:
        question = "What kind of design? (logo / poster / UI)"
    elif "analysis" in query:
        question = "Which analysis? (data / business / performance)"
    else:
        question = "Can you please clarify your request?"

    print("AI Agent:", question)
    user_answer = input("User: ")

    return {
        "clarification": user_answer,
        "is_ambiguous": False
    }


In [ ]:
def execute_task(state: AgentState):
    final_task = state["user_query"]

    if state["clarification"]:
        final_task += f" related to {state['clarification']}"

    print("\nAI Agent: Task is clear.")
    print("AI Agent: Executing task...")
    print("Final Task:", final_task)

    return {}


In [ ]:
graph = StateGraph(AgentState)

graph.add_node("detect_ambiguity", detect_ambiguity)
graph.add_node("ask_clarification", ask_clarification)
graph.add_node("execute_task", execute_task)

graph.set_entry_point("detect_ambiguity")

graph.add_conditional_edges(
    "detect_ambiguity",
    lambda state: "ask_clarification" if state["is_ambiguous"] else "execute_task",
)

graph.add_edge("ask_clarification", "execute_task")
graph.add_edge("execute_task", END)

agent = graph.compile()


In [ ]:
user_input = input("User: ")

agent.invoke({
    "user_query": user_input,
    "clarification": None,
    "is_ambiguous": False
})


User: create  a report
AI Agent: What type of report do you want? (sales / academic / project)
User: sales

AI Agent: Task is clear.
AI Agent: Executing task...
Final Task: create  a report related to sales


{'user_query': 'create  a report',
 'clarification': 'sales',
 'is_ambiguous': False}